### Import relevant packages

In [20]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd

### Pull APIs and Extract Relevant Data

In [ ]:
load_dotenv(dotenv_path="/Users/stella/Documents/DSProj/setlist-predictor/.env")
API_KEY = os.getenv("SETLISTFM_API_KEY")
HEADERS = {
    "x-api-key": API_KEY,
    "Accept": "application/json"
}

response = requests.get(
    "https://api.setlist.fm/rest/1.0/search/artists",
    headers=HEADERS,
    params={"artistName": "The Weeknd"}
)
print(response.status_code)
data = response.json()
data

200


{'type': 'artists',
 'itemsPerPage': 30,
 'page': 1,
 'total': 198,
 'artist': [{'mbid': '46c6c206-87f8-4d89-8037-b8383f60a5fc',
   'name': 'Belly & The Weeknd',
   'sortName': 'Belly & Weeknd, The',
   'disambiguation': '',
   'url': 'https://www.setlist.fm/setlists/belly-and-the-weeknd-339d54a1.html'},
  {'mbid': 'e92a186e-f432-456a-a38a-32a69acf5579',
   'name': 'Belly feat. The Weeknd',
   'sortName': 'Belly feat. Weeknd, The',
   'disambiguation': '',
   'url': 'https://www.setlist.fm/setlists/belly-feat-the-weeknd-2bc610d2.html'},
  {'mbid': 'e4a368ad-3a7c-4535-8699-f4b0d626464c',
   'name': 'Belly, The Weeknd & Young Thug',
   'sortName': 'Belly, Weeknd, The & Young Thug',
   'disambiguation': '',
   'url': 'https://www.setlist.fm/setlists/belly-the-weeknd-and-young-thug-bf3f126.html'},
  {'mbid': '36d67966-f288-426b-aa54-ca6ca5c07aba',
   'name': 'Beyoncé feat. The Weeknd',
   'sortName': 'Beyoncé feat. Weeknd, The',
   'disambiguation': '',
   'url': 'https://www.setlist.fm/se

In [ ]:
def find_artist_mbid(artist_name, headers, exact_match=True):
    page = 1
    while True:
        response = requests.get(
            "https://api.setlist.fm/rest/1.0/search/artists",
            headers=headers,
            params={"artistName": artist_name, "p": page}
        )
        
        if response.status_code != 200:
            print(f"Stopped — status {response.status_code} on page {page}")
            print(response.json())
            break
        
        data = response.json()
        
        if "total" not in data:
            print(f"Unexpected response shape on page {page}:")
            print(data)
            break
        
        for artist in data.get("artist", []):
            if exact_match and artist["name"].lower() == artist_name.lower():
                return artist["mbid"], artist["name"]
            elif not exact_match and artist_name.lower() in artist["name"].lower():
                print(artist["name"], "-", artist["mbid"])
        
        total_pages = -(-data["total"] // data["itemsPerPage"])
        if page >= total_pages:
            break
        page += 1
        
        import time
        time.sleep(0.5)
    
    return None, None

mbid, name = find_artist_mbid("The Weeknd", HEADERS)
print(mbid, name)

c8b03190-306c-4120-bb0b-6f2ebfc06ea9 The Weeknd


In [ ]:
mbid = "c8b03190-306c-4120-bb0b-6f2ebfc06ea9"

setlists = []
page = 1

while True:
    response = requests.get(
        f"https://api.setlist.fm/rest/1.0/artist/{mbid}/setlists",
        headers=HEADERS,
        params={"p": page}
    )
    if response.status_code != 200:
        print(f"Stopped at page {page}, status {response.status_code}")
        break
    
    page_data = response.json()
    setlists.extend(page_data.get("setlist", []))
    
    total_pages = -(-page_data["total"] // page_data["itemsPerPage"])
    print(f"Pulled page {page}/{total_pages}")
    
    if page >= total_pages:
        break
    page += 1
    
    import time
    time.sleep(0.5)

Pulled page 1/29
Pulled page 2/29
Pulled page 3/29
Pulled page 4/29
Pulled page 5/29
Pulled page 6/29
Pulled page 7/29
Pulled page 8/29
Pulled page 9/29
Pulled page 10/29
Pulled page 11/29
Pulled page 12/29
Pulled page 13/29
Pulled page 14/29
Pulled page 15/29
Pulled page 16/29
Pulled page 17/29
Pulled page 18/29
Pulled page 19/29
Pulled page 20/29
Pulled page 21/29
Pulled page 22/29
Pulled page 23/29
Pulled page 24/29
Pulled page 25/29
Pulled page 26/29
Pulled page 27/29
Pulled page 28/29
Pulled page 29/29


In [ ]:
print(len(setlists)) 
setlists[3]

572


{'id': '2340708b',
 'versionId': 'g7b04c218',
 'eventDate': '12-07-2026',
 'lastUpdated': '2026-07-15T17:14:03.273+0000',
 'artist': {'mbid': 'c8b03190-306c-4120-bb0b-6f2ebfc06ea9',
  'name': 'The Weeknd',
  'sortName': 'Weeknd, The',
  'disambiguation': 'Canadian R&B singer',
  'url': 'https://www.setlist.fm/setlists/the-weeknd-5bd26bb4.html'},
 'venue': {'id': '3bd6c4d8',
  'name': 'Stade de France',
  'city': {'id': '2980916',
   'name': 'Saint-Denis',
   'state': 'Île-de-France',
   'stateCode': '11',
   'coords': {'lat': 48.933, 'long': 2.367},
   'country': {'code': 'FR', 'name': 'France'}},
  'url': 'https://www.setlist.fm/venue/stade-de-france-saint-denis-france-3bd6c4d8.html'},
 'tour': {'name': 'After Hours Til Dawn'},
 'sets': {'set': [{'song': [{'name': 'Baptized in Fear', 'info': 'Shortened'},
     {'name': 'Open Hearts', 'info': 'Shortened'},
     {'name': 'Wake Me Up', 'info': 'Shortened'},
     {'name': 'After Hours', 'info': 'Shortened'},
     {'name': 'Starboy'},
    

In [22]:
rows = []
for show in setlists:
    if not show["sets"]["set"]:
        continue
    
    for set_block in show["sets"]["set"]:
        set_name = set_block.get("name", "main")
        for song in set_block.get("song", []):
            rows.append({
                "event_date": show["eventDate"],
                "venue": show["venue"]["name"],
                "city": show["venue"]["city"]["name"],
                "country": show["venue"]["city"]["country"]["name"],
                "tour": show.get("tour", {}).get("name", None),
                "set_name": set_name,
                "song_name": song.get("name"),
                "is_cover": "cover" in song,
                "info": song.get("info", None)
            })

df = pd.DataFrame(rows)
df["event_date"] = pd.to_datetime(df["event_date"], format="%d-%m-%Y")
df.head()

,event_date,venue,city,country,tour,set_name,song_name,is_cover,info
0,2026-07-12,Stade de France,Saint-Denis,France,After Hours Til Dawn,main,Baptized in Fear,False,Shortened
1,2026-07-12,Stade de France,Saint-Denis,France,After Hours Til Dawn,main,Open Hearts,False,Shortened
2,2026-07-12,Stade de France,Saint-Denis,France,After Hours Til Dawn,main,Wake Me Up,False,Shortened
3,2026-07-12,Stade de France,Saint-Denis,France,After Hours Til Dawn,main,After Hours,False,Shortened
4,2026-07-12,Stade de France,Saint-Denis,France,After Hours Til Dawn,main,Starboy,False,NaN


In [23]:
df.to_csv("../data/the_weeknd_setlists.csv", index=False)